In [119]:
import asyncio
import xmlschema
import networkx as nx
import polars as pl
from itertools import zip_longest

In [126]:
from ggblab import GeoGebra
ggb = await GeoGebra().init(use_vscode=True)

Using local cached file: xsd/common.xsd


<IPython.core.display.JSON object>

In [127]:
%pwd

'/Users/manabu/work/ggblab/examples'

In [5]:
%cd examples/

/Users/manabu/work/ggblab/examples


In [128]:
# ggb.file.load('eg10_slider2.ggb')
ggb.file.load('2025_06_08.ggb')

In [129]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [8]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [84]:
r = await ggb.function('getXML', ['b'])
print(r)

<element type="numeric" label="b">
	<value val="7"/>
	<slider min="0" max="8" absoluteScreenLocation="true" width="200" x="51.630259376842744" y="180.23697406231565" fixed="false" horizontal="true" showAlgebra="true"/>
	<lineStyle thickness="10" type="0" typeHidden="1"/>
	<show object="true" label="true"/>
	<objColor r="0" g="0" b="0" alpha="0.10000000149011612"/>
	<layer val="9"/>
	<labelMode val="1"/>
	<animation step="1" type="0" playing="false"/>
</element>



In [89]:
r = await ggb.function('getXML', ['b'])
o = ggb.file.ggb_schema.decode(r)
o['value'][0]['@val'] = '0'
x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
r = await ggb.function('evalXML' , [x])
r

In [130]:

df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)
df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
u32,str,str,str,str,str,u32,bool,bool,bool
0,"""O_{2}""","""point""",null,"""O_{2} = (-2.1, -0.1)""",null,9,true,true,false
1,"""A""","""point""",null,"""A = (10, 0)""",null,2,true,true,false
2,"""B""","""point""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",null,9,true,false,false
3,"""c""","""circle""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",null,9,true,false,false
4,"""n""","""line""","""Line(O_{2}, A)""","""n: y = -0.1""",null,4,false,false,false
5,"""O'""","""point""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",null,4,false,true,false
6,"""p""","""circle""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",null,4,false,false,false
7,"""j""","""line""","""Tangent(A, c)""","""j: -5.8x - 7.6y = -58.2""","""$a+c$""",2,true,false,false
8,"""l""","""line""","""Tangent(A, c)""","""l: 5.9x - 7.5y = 59.4""",null,2,true,false,false


In [131]:
p = ConstructionTreeParser(df)
g1 = p.parse()
nx.write_network_text(g1)

╟── O_{2}
╎   ├─╼ B ╾ A
╎   │   └─╼ c ╾ O_{2}
╎   │       ├─╼ j ╾ A
╎   │       │   ├─╼ q ╾ l
╎   │       │   ├─╼ r ╾ l
╎   │       │   │   └─╼ O_{1}
╎   │       │   │       ├─╼ t ╾ l
╎   │       │   │       │   ├─╼ F_{1} ╾ l
╎   │       │   │       │   │   ├─╼ h_1 ╾ O_{1}
╎   │       │   │       │   │   │   ├─╼ p_2 ╾ O_{2}, c
╎   │       │   │       │   │   │   │   ├─╼ q_2 ╾ O_{1}
╎   │       │   │       │   │   │   │   │   └─╼ N ╾ p_2
╎   │       │   │       │   │   │   │   │       └─╼ d_2 ╾ O_{2}
╎   │       │   │       │   │   │   │   │           └─╼ T ╾ c
╎   │       │   │       │   │   │   │   ├─╼ r_2 ╾ O_{1}
╎   │       │   │       │   │   │   │   │   └─╼ M ╾ p_2
╎   │       │   │       │   │   │   │   │       └─╼ b_2 ╾ O_{2}
╎   │       │   │       │   │   │   │   │           └─╼ F_{2} ╾ c
╎   │       │   │       │   │   │   │   │               ├─╼ γ ╾ A, O_{2}
╎   │       │   │       │   │   │   │   │               ├─╼ t2 ╾ A, O_{2}
╎   │       │   │       │   │   │   │   │   

In [132]:
p.df

Sequence,Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary,DependsOn
u32,str,str,str,str,str,u32,bool,bool,bool,list[str]
0,"""O_{2}""","""point""",null,"""O_{2} = (-2.1, -0.1)""",null,9,true,true,false,[]
1,"""A""","""point""",null,"""A = (10, 0)""",null,2,true,true,false,[]
2,"""B""","""point""","""Point(Line(O_{2}, A))""","""B = (-9.6, -0.2)""",null,9,true,false,false,"[""A"", ""O_{2}""]"
3,"""c""","""circle""","""Circle(O_{2}, B)""","""c: (x + 2.1)² + (y + 0.1)² = 5…",null,9,true,false,false,"[""A"", ""B"", ""O_{2}""]"
4,"""n""","""line""","""Line(O_{2}, A)""","""n: y = -0.1""",null,4,false,false,false,"[""A"", ""O_{2}""]"
5,"""O'""","""point""","""Midpoint(O_{2}, A)""","""O' = (3.9, 0)""",null,4,false,true,false,"[""A"", ""O_{2}""]"
6,"""p""","""circle""","""Circle(O', A)""","""p: (x - 3.9)² + (y + 0)² = 36.…",null,4,false,false,false,"[""A"", ""O'"", ""O_{2}""]"
7,"""j""","""line""","""Tangent(A, c)""","""j: -5.8x - 7.6y = -58.2""","""$a+c$""",2,true,false,false,"[""A"", ""B"", … ""c""]"
8,"""l""","""line""","""Tangent(A, c)""","""l: 5.9x - 7.5y = 59.4""",null,2,true,false,false,"[""A"", ""B"", … ""c""]"


In [166]:
await ggb.listen('k', True)
await ggb.listen('n_1', True)

{}

In [141]:
await ggb.listen('a', False)
await ggb.listen('b', False)

{}

In [167]:
ggb.comm.shared_objects

{}

In [162]:
await ggb.function("getVersion")

'5.2.909.9'

In [16]:
import ipywidgets as widgets
label1 = widgets.Label(value=ggb.comm.shared_objects['n'])
label2 = widgets.Label(value=ggb.comm.shared_objects['m'])
display(label1, label2)

Label(value='n = 6')

Label(value='m = 0')

In [146]:
async def on_shared_update1(changes):
    # await asyncio.sleep(0)
    # label1.value = changes['a']
    n = int(changes['k'].split()[2])
    # await asyncio.sleep(0)
    await ggb.function("setLayerVisible", list(zip_longest(range(9), [True]*n, fillvalue=False)))
    r = await ggb.function('getXML', ['n_1'])
    o = ggb.file.ggb_schema.decode(r)
    o['value'][0]['@val'] = '0'
    x = xmlschema.etree_tostring(ggb.file.ggb_schema.encode(o, 'element'))
    r = await ggb.function('evalXML' , [x])

In [147]:
ggb.comm.remove_shared_listener(on_shared_update1)
ggb.comm.add_shared_listener(on_shared_update1)

True

In [148]:
async def on_shared_update2(changes):
    # await asyncio.sleep(0)
    # label2.value = changes['b']
    m = int(changes['n_1'].split()[2])
    n = int(ggb.comm.shared_objects['k'].split()[2])
    l = df.filter(pl.col("Layer") == n)["Name"].to_list()
    # m = int(ggb.comm.shared_objects['m'].split()[2])
    # list(zip_longest(l, [True]*m, fillvalue=False))
    await ggb.function("setVisible", list(zip_longest(l, [True]*m, fillvalue=False)))

In [149]:
ggb.comm.remove_shared_listener(on_shared_update2)
ggb.comm.add_shared_listener(on_shared_update2)

True

In [104]:
ggb.comm.clear_shared_listeners()

2

In [103]:
await ggb.function("getVersion")

'5.2.909.9'

In [21]:
await ggb.command("Midpoint[D, E]")

'J'

In [22]:
await ggb.command("Circle(J, D)")

'e'

In [24]:
%pwd

'/Users/manabu/work/ggblab/examples'

In [116]:
# ggb.file.source_file = 'eg10_slider2.ggb'

In [168]:
ggb.file.base64_buffer = await ggb.function("getBase64")

In [169]:
ggb.file.save(overwrite=True)

* 原則（教育観）:
    - 目的化: 再現は「結果」ではなく「理解（なぜその操作か）」を目的にする。
    - 予測→検証: 次に何が起きるか予測させてから操作させる。
    - 説明要求: 手順ごとに短い理由説明（1文）を書かせる。
    - 変奏課題: パラメータを少し変えた課題で本質が移るか確認する。
    - 生成的課題: 「同じ発想で別の図形を作る」など転移を問う。
* ggblabで実装できる仕組み（短）:
    - 段階公開（layer slider）: 各レイヤーに「解説」「問い」「期待する操作」を紐付け、スライダーで段階的に提示。
    - 予測プロンプト: 各ステップの前に「次に何が起きる？」を表示し、回答を記録。
    - 説明入力欄: 学生が操作毎に短い説明を入力 → 教師や自動ルールでフィードバック。
    - 変化タスク自動化: DataFrame→コマンド生成を利用してパラメータをランダム化した派生課題を作る。
    - 操作ログ＋解析: 操作順・所要時間・試行回数をログ化して学習診断に使う。
    - 差分フィードバック: 学生構成と模範構成を比較して「次に直すべき一手」を提示。

In [29]:
await ggb.function("getVersion")

'5.2.909.9'